# Explore: Gemma 4 agent calibration evidence

**Author:** Moran Zavdi · **License:** Apache-2.0

Episode-level EDA over turn-compact JSONL from the **1,350**-episode campaign.
Fields follow `evidence/SCHEMA.md` (no raw reasoning / diffs).

**GitHub (source of truth):** https://github.com/Aeterna-Data-AI-ML-Research/Kaggle-Gemma-4-Developer-Agent-Competition

## Credits & related assets

**Author:** Moran Zavdi · **License:** Apache-2.0

### Canonical source (GitHub)
- **Repository (archived evidence release):** https://github.com/Aeterna-Data-AI-ML-Research/Kaggle-Gemma-4-Developer-Agent-Competition
- **Article (`PAPER.md`):** https://github.com/Aeterna-Data-AI-ML-Research/Kaggle-Gemma-4-Developer-Agent-Competition/blob/main/PAPER.md
- **Pages one-pager:** https://aeterna-data-ai-ml-research.github.io/Kaggle-Gemma-4-Developer-Agent-Competition/
- **Medium:** https://medium.com/@when1go2market

### This study on Kaggle
- **Dataset (required input):** [gemma4-agent-behavior-vs-accuracy](https://www.kaggle.com/datasets/moranzavdi/gemma4-agent-behavior-vs-accuracy) — `moranzavdi/gemma4-agent-behavior-vs-accuracy`
- **Reproduce notebook:** [reproduce-gemma4-agent-behavior-study](https://www.kaggle.com/code/moranzavdi/reproduce-gemma4-agent-behavior-study)
- **Explore notebook:** [explore-gemma4-agent-calibration](https://www.kaggle.com/code/moranzavdi/explore-gemma4-agent-calibration) ← *you are here*
- **Lessons notebook:** [gemma4-agent-lessons-for-competitors](https://www.kaggle.com/code/moranzavdi/gemma4-agent-lessons-for-competitors)

### Related paper-track assets (also by Moran Zavdi)
- **Dataset:** [gemma4-soundness-and-punica-bench](https://www.kaggle.com/datasets/moranzavdi/gemma4-soundness-and-punica-bench)
- **Notebook:** [from-local-validation-to-live-evaluation](https://www.kaggle.com/code/moranzavdi/from-local-validation-to-live-evaluation)

### Competition
- [Google — The Gemma 4 Developer Agent Competition](https://www.kaggle.com/competitions/gemma-4-developer-agent)

Please cite the GitHub release / `CITATION.cff` if you reuse numbers or code.


In [ ]:
from pathlib import Path
import subprocess, sys, tarfile

CANDIDATES = [
    Path('/kaggle/input/gemma4-agent-behavior-vs-accuracy'),
    Path('/kaggle/input/datasets/moranzavdi/gemma4-agent-behavior-vs-accuracy'),
    Path('.').resolve(),
    Path('..').resolve(),
]

def has_evidence(p: Path) -> bool:
    return (p / 'evidence' / 'phase1_calibration_900.jsonl').exists()

ROOT = next((p for p in CANDIDATES if has_evidence(p)), None)
if ROOT is None:
    # Kaggle sometimes ships directories as *.tar next to top-level files
    for base in CANDIDATES:
        if not base.exists():
            continue
        work = Path('/kaggle/working/gemma4_unpack')
        work.mkdir(parents=True, exist_ok=True)
        for tar_path in base.glob('*.tar'):
            with tarfile.open(tar_path) as tf:
                tf.extractall(work)
        if has_evidence(work):
            ROOT = work
            break
        if has_evidence(base):
            ROOT = base
            break
assert ROOT is not None, 'Attach dataset moranzavdi/gemma4-agent-behavior-vs-accuracy or clone the GitHub repo'
print('ROOT =', ROOT)

import json
from collections import Counter
import pandas as pd

def load_jsonl(p):
    return [json.loads(x) for x in Path(p).read_text().splitlines() if x.strip()]

p1 = load_jsonl(ROOT / 'evidence' / 'phase1_calibration_900.jsonl')
p2 = load_jsonl(ROOT / 'evidence' / 'phase2_guards_300.jsonl')
p3 = load_jsonl(ROOT / 'evidence' / 'phase3_thinkcap_150.jsonl')
print({'phase1': len(p1), 'phase2': len(p2), 'phase3': len(p3)})


In [ ]:
def episodes_frame(rows):
    return pd.DataFrame([{
        'phase': e['phase'],
        'arm': e['arm'],
        'seed': e['seed'],
        'task_id': e['task_id'],
        'wt': bool(e['pass']),
        'strict': bool(e['strict_pass']),
        'end_reason': e['end_reason'],
        'patch_submitted': bool(e['patch_submitted']),
        'thinking_budget': e['thinking_budget'],
        'thoughts_gt_1024': bool(e['thoughts_gt_1024']),
        'opened_gold_file_read': bool(e['opened_gold_file_read']),
        'n_turns': len(e.get('turn_summary') or []),
        'virtual_seconds': e['virtual_seconds'],
        'tool_calls': e['tool_calls'],
    } for e in rows])

df1 = episodes_frame(p1)
display(df1.groupby(['arm', 'seed']).agg(
    n=('task_id', 'count'),
    wt_rate=('wt', 'mean'),
    strict_rate=('strict', 'mean'),
    submit_rate=('patch_submitted', 'mean'),
    gold_open=('opened_gold_file_read', 'mean'),
).round(3))


In [ ]:
# Phase 3 Think-1024 (thinking_budget=1024) vs Phase-1 tr20 seed1 baseline (4096)
base = [e for e in p1 if e['arm'] == 'tr20' and e['seed'] == 1]
cap = p3
print('n base/cap', len(base), len(cap))

def summarize(label, rows):
    return {
        'label': label,
        'n': len(rows),
        'wt': sum(e['pass'] for e in rows),
        'strict': sum(e['strict_pass'] for e in rows),
        'submitted': sum(e['patch_submitted'] for e in rows),
        'thoughts_gt_1024': sum(e['thoughts_gt_1024'] for e in rows),
        'time_budget': sum(e['end_reason'] == 'time_budget' for e in rows),
    }

display(pd.DataFrame([summarize('tr20_s1_budget4096', base), summarize('think1024', cap)]))


In [ ]:
# Localization was usually already done before fatal timeouts
fatal = [e for e in p1 if e['end_reason'] == 'time_budget' and not e['pass'] and e.get('patch_chars', 0) == 0]
print('empty-diff time_budget fails:', len(fatal))
print('  opened gold via read:', sum(e['opened_gold_file_read'] for e in fatal))
print('  opened all gold files:', sum(e['opened_all_gold_files'] for e in fatal))
print('end_reason distribution (phase1):')
print(Counter(e['end_reason'] for e in p1).most_common())


## DS takeaway

Behavioral knobs (thinking budget, submit guards) can slash long thoughts and raise submit rate while **working-tree accuracy stays flat**. Pair every ablation with task-level WT + bootstrap CIs — run `01_reproduce_tables.ipynb`.


## Next steps & credits

| Go next | Link |
| --- | --- |
| GitHub repo | https://github.com/Aeterna-Data-AI-ML-Research/Kaggle-Gemma-4-Developer-Agent-Competition |
| Dataset | https://www.kaggle.com/datasets/moranzavdi/gemma4-agent-behavior-vs-accuracy |
| Reproduce | https://www.kaggle.com/code/moranzavdi/reproduce-gemma4-agent-behavior-study |
| Explore | https://www.kaggle.com/code/moranzavdi/explore-gemma4-agent-calibration |
| Lessons | https://www.kaggle.com/code/moranzavdi/gemma4-agent-lessons-for-competitors |
| Paper-track dataset | https://www.kaggle.com/datasets/moranzavdi/gemma4-soundness-and-punica-bench |
| Paper-track notebook | https://www.kaggle.com/code/moranzavdi/from-local-validation-to-live-evaluation |
| Competition | https://www.kaggle.com/competitions/gemma-4-developer-agent |
| Pages article | https://aeterna-data-ai-ml-research.github.io/Kaggle-Gemma-4-Developer-Agent-Competition/ |
| Medium | https://medium.com/@when1go2market |

Written by **Moran Zavdi**. If this notebook helped you, please star/cite the GitHub evidence release and credit the linked datasets/notebooks above.
